# Part 1: Pipelines for quick inference

A **pipeline** bundles a tokenizer, a model and the code that turns scores into labels. In this part you use two, and then do by hand what one of them does for you.

In [1]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from transformers import pipeline

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A ready-made sentiment classifier

The first call downloads the model from the Hub and keeps a copy on your computer. After that it loads from the copy.

In [2]:
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
sentences = [
    "A warm, funny and beautifully acted film.",
    "Dull, and far too long.",
    "Not bad at all.",
    "The plot was thin, but I loved every minute.",
]
for sentence in sentences:
    print(sentence, classifier(sentence))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1298.96it/s]


A warm, funny and beautifully acted film. [{'label': 'POSITIVE', 'score': 0.9998804330825806}]
Dull, and far too long. [{'label': 'NEGATIVE', 'score': 0.9998080134391785}]
Not bad at all. [{'label': 'POSITIVE', 'score': 0.99928218126297}]
The plot was thin, but I loved every minute. [{'label': 'POSITIVE', 'score': 0.9996179342269897}]


In [3]:
print(type(classifier.model).__name__)
print(type(classifier.tokenizer).__name__)
print(classifier(["great", "awful"]))  # a list in, a list out

DistilBertForSequenceClassification
BertTokenizer
[{'label': 'POSITIVE', 'score': 0.9998630285263062}, {'label': 'NEGATIVE', 'score': 0.9997954964637756}]


## What the pipeline does for you

Three steps: turn text into token ids, run the model to get **logits**, and turn the largest logit into a label name.

In [4]:
import torch

tokenizer, model = classifier.tokenizer, classifier.model
batch = tokenizer(["Dull, and far too long."], return_tensors="pt")
with torch.no_grad():
    logits = model(**batch).logits
print("logits", logits)
probabilities = torch.softmax(logits, dim=-1)
print("probabilities", probabilities)
best = probabilities.argmax(dim=-1).item()
print("label", model.config.id2label[best], probabilities[0, best].item())

logits tensor([[ 4.7348, -3.8231]])
probabilities tensor([[9.9981e-01, 1.9199e-04]])
label NEGATIVE 0.9998080134391785


## A pretrained model knows language

The `fill-mask` pipeline asks a base model to guess a hidden word. Nobody taught it this task. It learned it by reading a very large amount of text.

In [5]:
filler = pipeline("fill-mask", model="distilbert/distilbert-base-uncased")
for guess in filler("the film was [MASK] and i loved it.")[:3]:
    print(round(guess["score"], 3), guess["token_str"])

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 775.82it/s]


0.086 fantastic
0.066 funny
0.053 great


## Try it

1. Give the sentiment classifier a sentence with sarcasm. Does it get it right?
2. Give it a sentence about something other than a film. Is it still sensible?
3. Change the fill-mask sentence to one from a field you know. What does it guess?